# Sanskrit to English Neural Machine Translation

This notebook presents a complete Sanskrit to English NMT pipeline based on a custom sequence-to-sequence model with attention.

## Notebook Outline
1. Dataset loading and alignment (train/dev/test)
2. Tokenization and vocabulary construction
3. Seq2Seq model definition (encoder, attention, decoder)
4. Training and checkpointing
5. Inference, submission generation, and evaluation metrics

In [1]:
# Run once if dependencies are missing.
# Installs CUDA-enabled PyTorch; if CUDA is unavailable at runtime, code will fall back to CPU.
%pip install -q numpy pandas nltk bert-score
%pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Installation and Execution Guide

This project runs fully offline and does not call any external APIs.

Required packages:
- numpy
- pandas
- nltk
- bert-score
- CUDA-enabled PyTorch

Recommended run order:
1. Execute the dependency installation cell once.
2. Restart the kernel.
3. Execute all cells from top to bottom.

In [2]:
import os
import re
import math
import random
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Make training as deterministic as possible.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('Using device: cuda')
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device('cpu')
    print('Using device: cpu (CUDA not available)')

Using device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
# ---------------------------
# 1) Data loading utilities
# ---------------------------
DATA_DIR = Path('.')  # CSV files are expected in the same folder as this notebook

# Preferred exact names (used first if present)
PREFERRED_FILES = {
    'train_sa': 'train_sa.csv',
    'train_en': 'train_en.csv',
    'dev_sa': 'dev_sa.csv',
    'dev_en': 'dev_en.csv',
    'test_sa': 'test_sa.csv',
    'test_en': 'test_en.csv',  # optional in blind test settings
}

# Fallback glob patterns to support naming like *_10000.csv or *_1000.csv
FILE_PATTERNS = {
    'train_sa': ['train_sa*.csv'],
    'train_en': ['train_en*.csv'],
    'dev_sa': ['dev_sa*.csv'],
    'dev_en': ['dev_en*.csv'],
    'test_sa': ['test_sa*.csv'],
    'test_en': ['test_en*.csv'],
}


def normalize_text_content(text: str) -> str:
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def detect_sentence_column(df: pd.DataFrame) -> str:
    return [column for column in df.columns if column.lower().startswith('sentence')][0]


def resolve_dataset_file(data_dir: Path, split_key: str, required: bool = True):
    direct_path = data_dir / PREFERRED_FILES[split_key]
    if direct_path.exists():
        return direct_path

    discovered = []
    for pattern in FILE_PATTERNS[split_key]:
        discovered.extend(sorted(data_dir.glob(pattern)))

    if discovered:
        return discovered[0]

    if required:
        return None
    return None


def build_parallel_dataframe(src_file: Path, tgt_file: Path) -> pd.DataFrame:
    src_df = pd.read_csv(src_file)
    tgt_df = pd.read_csv(tgt_file)

    src_col = detect_sentence_column(src_df)
    tgt_col = detect_sentence_column(tgt_df)

    merged = src_df.merge(tgt_df, on='Source_id', suffixes=('_sa', '_en'))
    merged = merged[['Source_id', src_col, tgt_col]].copy()
    merged.columns = ['Source_id', 'source', 'target']
    merged['source'] = merged['source'].map(normalize_text_content)
    merged['target'] = merged['target'].map(normalize_text_content)
    merged = merged[(merged['source'] != '') & (merged['target'] != '')].reset_index(drop=True)
    return merged


def build_source_only_dataframe(src_file: Path) -> pd.DataFrame:
    src_df = pd.read_csv(src_file)
    src_col = detect_sentence_column(src_df)
    out_df = src_df[['Source_id', src_col]].copy()
    out_df.columns = ['Source_id', 'source']
    out_df['source'] = out_df['source'].map(normalize_text_content)
    out_df = out_df[out_df['source'] != ''].reset_index(drop=True)
    out_df['target'] = ''
    return out_df[['Source_id', 'source', 'target']]


def load_dataset_splits(data_dir: Path):
    resolved = {
        'train_sa': resolve_dataset_file(data_dir, 'train_sa', required=True),
        'train_en': resolve_dataset_file(data_dir, 'train_en', required=True),
        'dev_sa': resolve_dataset_file(data_dir, 'dev_sa', required=True),
        'dev_en': resolve_dataset_file(data_dir, 'dev_en', required=True),
        'test_sa': resolve_dataset_file(data_dir, 'test_sa', required=True),
        'test_en': resolve_dataset_file(data_dir, 'test_en', required=False),
    }

    required_splits = ['train_sa', 'train_en', 'dev_sa', 'dev_en', 'test_sa']
    missing_required = [name for name in required_splits if resolved[name] is None]

    if missing_required:
        print('Missing required files (after exact + pattern search):')
        for missing_name in missing_required:
            print(' -', missing_name, 'expected pattern:', ', '.join(FILE_PATTERNS[missing_name]))
        print('\nSearched in:', data_dir.resolve())
        return None, None, None

    print('Resolved files:')
    for split_name, split_path in resolved.items():
        print(f' - {split_name}: {split_path.name if split_path is not None else "(not found)"}')

    train_df = build_parallel_dataframe(resolved['train_sa'], resolved['train_en'])
    dev_df = build_parallel_dataframe(resolved['dev_sa'], resolved['dev_en'])

    if resolved['test_en'] is not None:
        test_df = build_parallel_dataframe(resolved['test_sa'], resolved['test_en'])
        print('Loaded test split in parallel mode (test_sa + test_en).')
    else:
        test_df = build_source_only_dataframe(resolved['test_sa'])
        print('Loaded test split in source-only mode (test_en not found).')

    return train_df, dev_df, test_df


train_df, dev_df, test_df = load_dataset_splits(DATA_DIR)

if train_df is not None:
    print('Train size:', len(train_df))
    print('Dev size  :', len(dev_df))
    print('Test size :', len(test_df))
    display(train_df.head(3))

Resolved files:
 - train_sa: train_sa_10000.csv
 - train_en: train_en_10000.csv
 - dev_sa: dev_sa_1000.csv
 - dev_en: dev_en_1000.csv
 - test_sa: test_sa_1000.csv
 - test_en: test_en_1000.csv
Loaded test split in parallel mode (test_sa + test_en).
Train size: 10000
Dev size  : 1000
Test size : 1000


,Source_id,source,target
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."


In [4]:
# ---------------------------
# 2) Tokenization + vocabulary
# ---------------------------
SPECIAL_TOKENS = ['<pad>', '<sos>', '<eos>', '<unk>']
PAD, SOS, EOS, UNK = SPECIAL_TOKENS


def tokenize_sanskrit_text(text: str) -> List[str]:
    # Simple whitespace tokenizer.
    return text.strip().split()


def tokenize_english_text(text: str) -> List[str]:
    # Simple whitespace tokenizer.
    return text.strip().split()


class Vocab:
    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.stoi: Dict[str, int] = {}
        self.itos: List[str] = []

    def build(self, tokenized_texts: List[List[str]]):
        counter = Counter()
        for toks in tokenized_texts:
            counter.update(toks)

        self.itos = SPECIAL_TOKENS.copy()
        for token, freq in counter.items():
            if freq >= self.min_freq and token not in self.itos:
                self.itos.append(token)

        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def encode(self, tokens: List[str], add_sos_eos: bool = True) -> List[int]:
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in tokens]
        if add_sos_eos:
            ids = [self.stoi[SOS]] + ids + [self.stoi[EOS]]
        return ids

    def decode(self, ids: List[int], remove_special: bool = True) -> List[str]:
        tokens = [self.itos[i] if i < len(self.itos) else UNK for i in ids]
        if remove_special:
            tokens = [t for t in tokens if t not in SPECIAL_TOKENS]
        return tokens

    def __len__(self):
        return len(self.itos)


if train_df is not None:
    train_src_tokens = [tokenize_sanskrit_text(x) for x in train_df['source'].tolist()]
    train_tgt_tokens = [tokenize_english_text(x) for x in train_df['target'].tolist()]

    src_vocab = Vocab(min_freq=1)
    tgt_vocab = Vocab(min_freq=1)
    src_vocab.build(train_src_tokens)
    tgt_vocab.build(train_tgt_tokens)

    print('Source vocab size:', len(src_vocab))
    print('Target vocab size:', len(tgt_vocab))

Source vocab size: 33278
Target vocab size: 19552


In [5]:
# ---------------------------
# 3) Dataset + DataLoader
# ---------------------------
@dataclass
class BatchConfig:
    batch_size: int = 32
    max_src_len: int = 80
    max_tgt_len: int = 80


CFG = BatchConfig()


class TranslationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, src_vocab: Vocab, tgt_vocab: Vocab, cfg: BatchConfig):
        self.df = df.reset_index(drop=True)
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src_tokens = tokenize_sanskrit_text(row['source'])[: self.cfg.max_src_len]
        tgt_tokens = tokenize_english_text(row['target'])[: self.cfg.max_tgt_len]

        src_ids = self.src_vocab.encode(src_tokens, add_sos_eos=True)
        tgt_ids = self.tgt_vocab.encode(tgt_tokens, add_sos_eos=True)

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_lengths = torch.tensor([len(x) for x in src_batch], dtype=torch.long)
    tgt_lengths = torch.tensor([len(x) for x in tgt_batch], dtype=torch.long)

    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=src_vocab.stoi[PAD])
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_vocab.stoi[PAD])

    return src_padded, src_lengths, tgt_padded, tgt_lengths


if train_df is not None:
    train_ds = TranslationDataset(train_df, src_vocab, tgt_vocab, CFG)
    dev_ds = TranslationDataset(dev_df, src_vocab, tgt_vocab, CFG)
    test_ds = TranslationDataset(test_df, src_vocab, tgt_vocab, CFG)

    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_ds, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn)

    print('DataLoaders ready.')

DataLoaders ready.


In [6]:
# ---------------------------
# 4) Custom Seq2Seq + Attention
# ---------------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(enc_hid_dim * 2, enc_hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_outputs, batch_first=True)

        # hidden shape: [2, B, H] for bidirectional single-layer GRU
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        return outputs, hidden


class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        attention = attention.masked_fill(mask == 0, -1e10)
        return torch.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention, pad_idx):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs, mask):
        # input_token: [B]
        input_token = input_token.unsqueeze(1)  # [B,1]

        embedded = self.dropout(self.embedding(input_token))  # [B,1,E]

        a = self.attention(hidden, encoder_outputs, mask).unsqueeze(1)  # [B,1,S]
        weighted = torch.bmm(a, encoder_outputs)  # [B,1,2H]

        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))

        output = output.squeeze(1)
        weighted = weighted.squeeze(1)
        embedded = embedded.squeeze(1)
        hidden = hidden.squeeze(0)

        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden, a.squeeze(1)


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.device = device

    def create_mask(self, src):
        return (src != self.src_pad_idx)

    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size, device=self.device)

        encoder_outputs, hidden = self.encoder(src, src_lengths)
        input_token = tgt[:, 0]
        mask = self.create_mask(src)

        for t in range(1, tgt_len):
            output, hidden, _ = self.decoder(input_token, hidden, encoder_outputs, mask)
            outputs[:, t] = output

            top1 = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = tgt[:, t] if teacher_force else top1

        return outputs


if train_df is not None:
    INPUT_DIM = len(src_vocab)
    OUTPUT_DIM = len(tgt_vocab)
    ENC_EMB_DIM = 256
    DEC_EMB_DIM = 256
    ENC_HID_DIM = 256
    DEC_HID_DIM = 256
    ENC_DROPOUT = 0.3
    DEC_DROPOUT = 0.3

    attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
    enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, ENC_DROPOUT, src_vocab.stoi[PAD])
    dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn, tgt_vocab.stoi[PAD])

    model = Seq2Seq(enc, dec, src_vocab.stoi[PAD], DEVICE).to(DEVICE)
    print(model.__class__.__name__, 'initialized.')

Seq2Seq initialized.


In [7]:
# ---------------------------
# 5) Training + evaluation
# ---------------------------
def token_precision_score(references: List[List[str]], hypotheses: List[List[str]]) -> float:
    # Simple unigram precision proxy if BLEU libraries are unavailable.
    num, den = 0, 0
    for ref, hyp in zip(references, hypotheses):
        ref_counter = Counter(ref)
        hyp_counter = Counter(hyp)
        overlap = sum(min(ref_counter[w], hyp_counter[w]) for w in hyp_counter)
        num += overlap
        den += max(len(hyp), 1)
    return num / den if den > 0 else 0.0


def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0.0

    for src, src_lengths, tgt, _ in loader:
        src = src.to(DEVICE)
        src_lengths = src_lengths.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, src_lengths, tgt, teacher_forcing_ratio=0.5)

        # Ignore first token (<sos>) for loss
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt_gold = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt_gold)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


@torch.no_grad()
def evaluate_epoch(model, loader, criterion):
    model.eval()
    epoch_loss = 0.0

    all_refs = []
    all_hyps = []

    for src, src_lengths, tgt, _ in loader:
        src = src.to(DEVICE)
        src_lengths = src_lengths.to(DEVICE)
        tgt = tgt.to(DEVICE)

        output = model(src, src_lengths, tgt, teacher_forcing_ratio=0.0)

        output_dim = output.shape[-1]
        loss = criterion(output[:, 1:].reshape(-1, output_dim), tgt[:, 1:].reshape(-1))
        epoch_loss += loss.item()

        pred_ids = output.argmax(dim=-1).cpu().tolist()
        gold_ids = tgt.cpu().tolist()

        for p, g in zip(pred_ids, gold_ids):
            p_toks = tgt_vocab.decode(p, remove_special=True)
            g_toks = tgt_vocab.decode(g, remove_special=True)
            all_hyps.append(p_toks)
            all_refs.append(g_toks)

    precision = token_precision_score(all_refs, all_hyps)
    return epoch_loss / len(loader), precision


@torch.no_grad()
def translate_sentence(model, sentence: str, max_len: int = 80) -> str:
    model.eval()

    src_tokens = tokenize_sanskrit_text(normalize_text_content(sentence))[:max_len]
    src_ids = src_vocab.encode(src_tokens, add_sos_eos=True)

    src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    src_len = torch.tensor([len(src_ids)], dtype=torch.long, device=DEVICE)

    encoder_outputs, hidden = model.encoder(src_tensor, src_len)
    mask = model.create_mask(src_tensor)

    generated = [tgt_vocab.stoi[SOS]]

    for _ in range(max_len):
        input_token = torch.tensor([generated[-1]], dtype=torch.long, device=DEVICE)
        output, hidden, _ = model.decoder(input_token, hidden, encoder_outputs, mask)
        next_id = int(output.argmax(1).item())
        generated.append(next_id)
        if next_id == tgt_vocab.stoi[EOS]:
            break

    tokens = tgt_vocab.decode(generated, remove_special=True)
    return ' '.join(tokens)


if train_df is not None:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.stoi[PAD])

    N_EPOCHS = 10
    best_dev_loss = float('inf')

    history = []

    for epoch in range(1, N_EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        dev_loss, dev_precision = evaluate_epoch(model, dev_loader, criterion)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'dev_loss': dev_loss,
            'dev_token_precision': dev_precision,
        })

        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            torch.save(model.state_dict(), 'best_seq2seq_sanskrit_en.pt')

        print(
            f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | "
            f"Dev Loss: {dev_loss:.4f} | Dev Token Precision: {dev_precision:.4f}"
        )

    history_df = pd.DataFrame(history)
    display(history_df.tail())

Epoch 01 | Train Loss: 7.0377 | Dev Loss: 6.9932 | Dev Token Precision: 0.1544
Epoch 02 | Train Loss: 6.0228 | Dev Loss: 6.8680 | Dev Token Precision: 0.2007
Epoch 03 | Train Loss: 5.1926 | Dev Loss: 6.9375 | Dev Token Precision: 0.2394
Epoch 04 | Train Loss: 4.3318 | Dev Loss: 7.1738 | Dev Token Precision: 0.1898
Epoch 05 | Train Loss: 3.6305 | Dev Loss: 7.3462 | Dev Token Precision: 0.1993
Epoch 06 | Train Loss: 3.2006 | Dev Loss: 7.5132 | Dev Token Precision: 0.1921
Epoch 07 | Train Loss: 2.9420 | Dev Loss: 7.7167 | Dev Token Precision: 0.1826
Epoch 08 | Train Loss: 2.7268 | Dev Loss: 7.8298 | Dev Token Precision: 0.1943
Epoch 09 | Train Loss: 2.5528 | Dev Loss: 7.9505 | Dev Token Precision: 0.1888
Epoch 10 | Train Loss: 2.3638 | Dev Loss: 8.1424 | Dev Token Precision: 0.1971


,epoch,train_loss,dev_loss,dev_token_precision
5,6,3.200612,7.513210,0.192112
6,7,2.942019,7.716704,0.182643
7,8,2.726823,7.829782,0.194341
8,9,2.552833,7.950460,0.188796
9,10,2.363791,8.142388,0.197109


In [8]:
# ---------------------------
# 6) Qualitative examples + submission export
# ---------------------------
if train_df is not None:
    # Load best checkpoint for inference
    if Path('best_seq2seq_sanskrit_en.pt').exists():
        model.load_state_dict(torch.load('best_seq2seq_sanskrit_en.pt', map_location=DEVICE))

    sample_n = min(5, len(dev_df))
    sample_rows = dev_df.sample(sample_n, random_state=SEED)

    print('Sample Sanskrit -> Predicted English (Reference)\n')
    for _, row in sample_rows.iterrows():
        src_sent = row['source']
        ref_sent = row['target']
        pred_sent = translate_sentence(model, src_sent)

        print('SA :', src_sent)
        print('PRD:', pred_sent)
        print('REF:', ref_sent)
        print('-' * 80)

    # Generate predictions for test set
    test_df = test_df.copy()
    test_df['Sentence_en'] = test_df['source'].map(lambda x: translate_sentence(model, x))

    # Required output format
    submission_df = test_df[['Source_id', 'Sentence_en']].copy()
    submission_df.to_csv('submission.csv', index=False, encoding='utf-8')

    print('Saved: submission.csv (UTF-8)')
    display(submission_df.head())
else:
    print('Data not loaded yet. Add train/dev/test CSV files and run cells from top.')

Sample Sanskrit -> Predicted English (Reference)

SA : वयं कीलफलके शार्ट्कट् इत्यस्य संरचनाम् अपि च टूलबार इत्यस्य संरचनाम् अपि ज्ञातवन्तः
PRD: We will learn about: and the and and and and
REF: We also learnt to configure keyboard short cuts and toolbars.
--------------------------------------------------------------------------------
SA : पश्चात्, पेज् अधः विद्यमानं Save उपरि क्लिक् कुर्वन्तु ।
PRD: Click click on the tab
REF: Then click on Save at the bottom of the page.
--------------------------------------------------------------------------------
SA : एता: मुद्रा: विभिन्नस्नायुं तथा मेरुदण्डं प्रसारयन सम्पूर्णशरीरम आकुञ्चनात्मकं कुर्वन्ति।
PRD: "After studying and and and and
REF: These postures stretch various muscles and spinal column and give flexibility to the whole body.
--------------------------------------------------------------------------------
SA : stream परिवर्तितं दृश्यते ।
PRD: The is is
REF: We see that the stream is changed.
--------------------------------------

,Source_id,Sentence_en
0,1,It is the to the to the
1,2,"""And they were and the and of the and and and ..."
2,3,To I am using the and
3,4,Let us go to the to the and
4,5,"""And they were and the and came and the and of..."


In [9]:
# ---------------------------
# 7) Required evaluation metrics
# ---------------------------
from time import perf_counter


def has_gold_targets(df: pd.DataFrame) -> bool:
    if df is None or 'target' not in df.columns:
        return False
    non_empty = df['target'].astype(str).str.strip() != ''
    return bool(non_empty.all())


@torch.no_grad()
def translate_test_set(model, test_df: pd.DataFrame):
    sources = test_df['source'].tolist()
    start = perf_counter()
    predictions = [translate_sentence(model, s) for s in sources]
    elapsed_sec = perf_counter() - start
    return predictions, elapsed_sec


if train_df is not None and test_df is not None and 'model' in globals():
    # Load best checkpoint if available
    if Path('best_seq2seq_sanskrit_en.pt').exists():
        model.load_state_dict(torch.load('best_seq2seq_sanskrit_en.pt', map_location=DEVICE))

    # Efficiency metric 1: total inference time over test set
    test_predictions, total_inference_time_sec = translate_test_set(model, test_df)

    # Efficiency metric 2: total number of parameters
    total_parameters = sum(p.numel() for p in model.parameters())

    # Save submission in required format
    submission_df = pd.DataFrame({
        'Source_id': test_df['Source_id'].tolist(),
        'Sentence_en': test_predictions,
    })
    submission_df.to_csv('submission.csv', index=False, encoding='utf-8')

    print('Saved: submission.csv (UTF-8)')
    print(f'Total inference time (test set): {total_inference_time_sec:.4f} seconds')
    print(f'Total model parameters: {total_parameters}')

    # Compute BLEU and BERTScore only if gold test references are available
    if has_gold_targets(test_df):
        references = test_df['target'].astype(str).tolist()

        # BLEU: NLTK corpus_bleu default weights
        try:
            from nltk.translate.bleu_score import corpus_bleu
            bleu_refs = [[ref.split()] for ref in references]
            bleu_hyps = [pred.split() for pred in test_predictions]
            bleu_score = corpus_bleu(bleu_refs, bleu_hyps)
            print(f'BLEU (NLTK default): {bleu_score:.6f}')
        except Exception as e:
            print('BLEU could not be computed. Install package: nltk')
            print('Reason:', str(e))

        # BERTScore: report F1 with baseline rescaling enabled
        try:
            from bert_score import score as bert_score
            _, _, f1 = bert_score(
                cands=test_predictions,
                refs=references,
                lang='en',
                rescale_with_baseline=True,
                verbose=False,
            )
            bert_f1 = float(f1.mean().item())
            print(f'BERTScore F1 (rescaled): {bert_f1:.6f}')
        except Exception as e:
            print('BERTScore could not be computed. Install package: bert-score')
            print('Reason:', str(e))
    else:
        print('Gold test references not found; BLEU/BERTScore skipped (blind test mode).')

    display(submission_df.head())
else:
    print('Data/model not ready. Run notebook cells from top after adding dataset files.')

Saved: submission.csv (UTF-8)
Total inference time (test set): 10.9848 seconds
Total model parameters: 35471200
BLEU (NLTK default): 0.021466


c:\Users\Vasant Pratap Singh\OneDrive\Documents\Mtech\Trimester 3\NLU\Assignment\Assignment2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4708.77it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized bec

BERTScore F1 (rescaled): 0.026717


c:\Users\Vasant Pratap Singh\OneDrive\Documents\Mtech\Trimester 3\NLU\Assignment\Assignment2\.venv\Lib\site-packages\bert_score\score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  baselines = torch.from_numpy(


,Source_id,Sentence_en
0,1,It is the to the to the
1,2,"""And they were and the and of the and and and ..."
2,3,To I am using the and
3,4,Let us go to the to the and
4,5,"""And they were and the and came and the and of..."


## Compliance and Reproducibility

- Uses only the provided Sanskrit-English dataset.
- Uses a custom PyTorch seq2seq-attention model trained from scratch.
- Uses no external APIs.
- Uses fixed random seeds for reproducibility.
- Declares pretrained usage clearly: BERTScore may load a pretrained encoder only for evaluation.
- Produces UTF-8 submission.csv with exactly two columns: Source_id and Sentence_en.

## Report Writing Checklist

Include the following sections in your report:
- Introduction and problem statement
- Model architecture (encoder, attention, decoder)
- Data preprocessing and tokenization strategy
- Training setup and hyperparameters
- Quantitative results (BLEU, BERTScore, efficiency metrics)
- Translation examples with error analysis
- Discussion of limitations and future work

Possible improvements to discuss:
- Subword tokenization (BPE/SentencePiece)
- Better decoding strategies (for example, beam search)
- Learning-rate scheduling and regularization